In [1]:
%cd ../..

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


In [6]:
from data.utils_strokerehab import PrimitiveLabelUtils, DataPaths
from lmms_eval.tasks.strokerehab import utils_primitives
import pandas as pd

In [7]:
df = pd.DataFrame(
    utils_primitives.load_strokerehab_primitives_dataset(patients="C00020")['test']
)

In [12]:
import os
def get_labels(path_l):
    path_l = os.path.join(DataPaths.RAW_LABEL_DIR, path_l)
    sequence = PrimitiveLabelUtils.convert_labels_to_action_sequence(path_l)
    return sequence

In [13]:
df['prims'] = df['path_l'].apply(get_labels)

In [ ]:
from collections import defaultdict
def sequences_in_window(sequence, window_s=8.0/15):
    """
    Count ordered tuples of primitive 'action's overlapping each window of width `window_s`.
    Windows advance by `window_s` (non-overlapping). Assumes `sequence` is sorted by start_time.
    Each element is a dict with keys: 'action', 'start_time', 'duration'.
    """
    if not sequence:
        return {}

    n = len(sequence)
    total_T = sequence[-1]['start_time'] + sequence[-1]['duration']
    ans = defaultdict(int)

    i = 0  # left pointer: first prim whose end_time > window_start
    t = 0.0
    while t < total_T:
        t0, t1 = t, min(t + window_s, total_T)

        # advance i so that sequence[i] still overlaps the window start
        while i < n and (sequence[i]['start_time'] + sequence[i]['duration']) <= t0:
            i += 1

        # collect all prims that overlap [t0, t1)
        j = i
        prims_in_window = []
        while j < n and sequence[j]['start_time'] < t1:
            # overlap is guaranteed by the i-advance and this bound
            prims_in_window.append(sequence[j]['action'])
            j += 1

        ans[tuple(prims_in_window)] += 1
        t += window_s
    return ans

In [23]:
from collections import Counter

all_counts = Counter()
for seq in df["prims"]:
    all_counts.update(sequences_in_window(seq))

print(all_counts)

Counter({('transport',): 2160, ('reach', 'transport'): 700, ('reach',): 667, ('reposition',): 544, ('idle',): 534, ('stabilize',): 446, ('transport', 'reposition'): 368, ('idle', 'reach'): 254, ('reposition', 'idle'): 247, ('reposition', 'idle', 'reach'): 200, ('transport', 'reach'): 192, ('transport', 'stabilize'): 177, ('stabilize', 'transport'): 139, ('transport', 'stabilize', 'reposition'): 96, ('transport', 'stabilize', 'transport'): 91, ('reposition', 'reach'): 86, ('stabilize', 'reposition'): 44, ('idle', 'reposition'): 34, ('reach', 'stabilize'): 26, ('reach', 'transport', 'reach'): 20, ('transport', 'reach', 'transport'): 20, ('stabilize', 'reach'): 20, ('reach', 'idle'): 14, ('stabilize', 'transport', 'stabilize'): 12, ('transport', 'stabilize', 'reach'): 8, ('reach', 'stabilize', 'transport'): 8, (): 8, ('stabilize', 'transport', 'reach'): 8, ('reach', 'reposition'): 6, ('reach', 'transport', 'stabilize'): 6, ('reach', 'idle', 'reach'): 4, ('transport', 'stabilize', 'transpo

In [ ]:
all_counts

In [26]:
all_counts

Counter({('transport',): 2160,
         ('reach', 'transport'): 700,
         ('reach',): 667,
         ('reposition',): 544,
         ('idle',): 534,
         ('stabilize',): 446,
         ('transport', 'reposition'): 368,
         ('idle', 'reach'): 254,
         ('reposition', 'idle'): 247,
         ('reposition', 'idle', 'reach'): 200,
         ('transport', 'reach'): 192,
         ('transport', 'stabilize'): 177,
         ('stabilize', 'transport'): 139,
         ('transport', 'stabilize', 'reposition'): 96,
         ('transport', 'stabilize', 'transport'): 91,
         ('reposition', 'reach'): 86,
         ('stabilize', 'reposition'): 44,
         ('idle', 'reposition'): 34,
         ('reach', 'stabilize'): 26,
         ('reach', 'transport', 'reach'): 20,
         ('transport', 'reach', 'transport'): 20,
         ('stabilize', 'reach'): 20,
         ('reach', 'idle'): 14,
         ('stabilize', 'transport', 'stabilize'): 12,
         ('transport', 'stabilize', 'reach'): 8,
     

In [27]:
transport_counts = {k: v for k, v in all_counts.items() if len(k) >= 1 and 'transport' == k[0]}

In [29]:
stabilize_counts = {k: v for k, v in all_counts.items() if len(k) >= 1 and 'stabilize' == k[0]}

In [28]:
transport_counts

{('transport',): 2160,
 ('transport', 'stabilize', 'transport'): 91,
 ('transport', 'reach'): 192,
 ('transport', 'reach', 'transport'): 20,
 ('transport', 'stabilize', 'reach', 'transport'): 2,
 ('transport', 'stabilize', 'reach'): 8,
 ('transport', 'reposition'): 368,
 ('transport', 'stabilize'): 177,
 ('transport', 'reach', 'transport', 'reposition'): 2,
 ('transport', 'stabilize', 'reposition'): 96,
 ('transport', 'reach', 'stabilize'): 2,
 ('transport', 'stabilize', 'transport', 'stabilize'): 2,
 ('transport', 'idle', 'reach'): 2,
 ('transport', 'stabilize', 'transport', 'reach'): 4,
 ('transport', 'reposition', 'idle'): 2,
 ('transport', 'stabilize', 'transport', 'stabilize', 'transport'): 2,
 ('transport', 'reposition', 'reach'): 2,
 ('transport', 'idle'): 2}

In [30]:
stabilize_counts

{('stabilize',): 446,
 ('stabilize', 'transport'): 139,
 ('stabilize', 'reposition'): 44,
 ('stabilize', 'reach'): 20,
 ('stabilize', 'transport', 'reach'): 8,
 ('stabilize', 'transport', 'stabilize'): 12,
 ('stabilize', 'reach', 'transport'): 4}

In [16]:
df['prims'].iloc[0][:5]

[{'action': 'idle', 'start_time': 0.0, 'duration': 3.167},
 {'action': 'reach', 'start_time': 3.167, 'duration': 1.617},
 {'action': 'stabilize', 'start_time': 4.784, 'duration': 5.167000000000001},
 {'action': 'transport', 'start_time': 9.951, 'duration': 3.6679999999999993},
 {'action': 'stabilize',
  'start_time': 13.619,
  'duration': 0.08300000000000018}]

In [ ]:
# If something 